# 03_pc_<agreement>_<source>_to_<target> — deterministic pipeline enforcement

Use this run-all-safe notebook to enforce approved contracts, apply deterministic quality checks, write outputs, and record metadata/lineage evidence.

**You edit**
- Dataset/table names and source/target kinds
- Deterministic transformation logic cell
- Target write mode and optional technical column choices

**This notebook produces**
- Contract-enforced output dataset
- DQ result records and run metadata
- Lineage and handover evidence

**Enforcement boundary**
- This notebook enforces only human-approved contracts and rules.
- AI does not decide pipeline outcomes.


## Segment 1: Load shared config and runtime context

What this does: loads environment config and builds notebook runtime context.

Functions used: `load_fabric_config`, `validate_notebook_name`, `build_runtime_context`, `generate_run_id`.

Output: runtime context for deterministic execution.


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit.environment_config import bootstrap_fabric_env, get_path, load_fabric_config
from fabricops_kit.runtime_context import (
    assert_notebook_name_valid,
    build_runtime_context,
    generate_run_id,
    validate_notebook_name,
)
from fabricops_kit.fabric_input_output import lakehouse_table_read, lakehouse_table_write, warehouse_read, warehouse_write
from pathlib import Path
from runpy import run_path
from pyspark.sql import functions as F
from fabricops_kit.data_contracts import (
    normalize_contract_dict,
    validate_contract_dict,
    extract_business_keys,
    extract_required_columns,
    get_executable_quality_rules,
    load_latest_approved_contract,
    build_contract_records,
)
from fabricops_kit.data_quality import assert_dq_passed, run_dq_rules, split_valid_and_quarantine, validate_dq_rules
from fabricops_kit.technical_audit_columns import add_audit_columns, add_datetime_features, add_hash_columns, default_technical_columns
from fabricops_kit.data_profiling import generate_metadata_profile, profile_dataframe_to_metadata
from fabricops_kit.data_product_metadata import build_dataset_run_record, build_quality_result_records
from fabricops_kit.data_lineage import build_lineage_from_notebook_code, build_lineage_handover_markdown, build_lineage_records, build_lineage_record_from_steps


In [ ]:
USE_SAMPLE_DATA = True
ENV_NAME = "dev"
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_KIND = "lakehouse"
TARGET_KIND = "lakehouse"
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else "TODO_target_table"
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "target_dataset"
WRITE_MODE = "overwrite"


In [ ]:
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name()
assert_notebook_name_valid()
if not USE_SAMPLE_DATA:
    bootstrap_fabric_env(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])
RUN_ID = generate_run_id(prefix="pc")
runtime_context = build_runtime_context(dataset_name=DATASET_NAME, environment=ENV_NAME, source_table=SOURCE_TABLE, target_table=TARGET_TABLE, run_id=RUN_ID)


## Load approved contract and fail fast
Load approved metadata-table contract immediately after source read and validate required columns before transformation steps.


## Segment 2: Load approved contract and source data

What this does: retrieves approved contract and reads source data from lakehouse/warehouse.

Functions used: `load_latest_approved_contract`, `get_path`, `lakehouse_table_read`, `warehouse_read`.

Output: source dataframe plus approved contract constraints.


In [ ]:
if USE_SAMPLE_DATA:
    contract_module = run_path("samples/end_to_end/minimal_source_contract.py")
    contract = normalize_contract_dict(contract_module["MINIMAL_SOURCE_CONTRACT"])
    contract_errors = validate_contract_dict(contract)
    if contract_errors:
        raise ValueError("Contract validation failed: " + " | ".join(contract_errors))
    if contract.get("status") != "approved":
        raise ValueError("Sample contract must be approved.")
else:
    metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)
    contract = load_latest_approved_contract(metadata_path, dataset_name=DATASET_NAME, object_name=SOURCE_TABLE, contract_type="source_input")


In [ ]:
if USE_SAMPLE_DATA:
    df_source = spark.read.option("header", True).csv("samples/end_to_end/minimal_source.csv", inferSchema=True)
elif SOURCE_KIND == "lakehouse":
    source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
required_columns = extract_required_columns(contract)
business_keys = extract_business_keys(contract)

missing = sorted(set(required_columns) - set(df_source.columns))
if missing:
    raise ValueError(f"Fail fast: missing required source columns: {missing}")


In [ ]:
df_transformed = (
    df_source.withColumn("status", F.trim(F.lower(F.col("status"))))
    .withColumn("email", F.trim(F.lower(F.col("email"))))
    .withColumn("amount", F.col("amount").cast("double"))
)


## Segment 3: Apply deterministic transforms and technical columns

What this does: applies required-column checks, user transform logic, and technical audit columns.

Functions used: `extract_required_columns`, `extract_business_keys`, `add_audit_columns`, `add_hash_columns`.

Output: curated dataframe ready for DQ enforcement.


In [ ]:
rules = get_executable_quality_rules(contract)
validate_dq_rules(rules)
if not rules and "DQ_RULES" in globals():
    rules = DQ_RULES.get(TARGET_TABLE, [])
    validate_dq_rules(rules)


In [ ]:
DATETIME_COLUMN = None
DATETIME_PREFIX = None
if DATETIME_COLUMN:
    df_transformed = add_datetime_features(
        df_transformed,
        datetime_column=DATETIME_COLUMN,
        prefix=DATETIME_PREFIX,
    )


In [ ]:
BUSINESS_KEYS = list(business_keys or [])
df_standard = add_audit_columns(df_transformed, run_id=RUN_ID)
if BUSINESS_KEYS:
    df_standard = add_hash_columns(
        df_standard,
        business_keys=BUSINESS_KEYS,
    )


In [ ]:
tech_cols = default_technical_columns()
required_output_columns = set(extract_required_columns(contract) or [])
actual_output_cols = set(c for c in df_standard.columns if c not in tech_cols)
if required_output_columns and required_output_columns != actual_output_cols:
    raise ValueError("Fail fast: output contract mismatch.")


## Segment 4: Enforce quality and publish outputs

What this does: runs approved DQ rules, writes target outputs, and records metadata evidence.

Functions used: `run_dq_rules`, `assert_dq_passed`, `lakehouse_table_write`/`warehouse_write`, `build_dataset_run_record`.

Output: persisted output plus quality/run evidence.


In [ ]:
dq_result = run_dq_rules(df_standard, table_name=TARGET_TABLE, rules=rules, fail_on_error=False)
display(dq_result)
df_valid, df_quarantine = split_valid_and_quarantine(df_standard, rules)
if USE_SAMPLE_DATA:
    output_dir = Path("samples/end_to_end/_output")
    output_dir.mkdir(parents=True, exist_ok=True)
    df_valid.toPandas().to_csv(output_dir / f"{TARGET_TABLE}.csv", index=False)
    df_quarantine.toPandas().to_csv(output_dir / f"{TARGET_TABLE}_QUARANTINE.csv", index=False)
    dq_result.toPandas().to_json(output_dir / "dq_result.jsonl", orient="records", lines=True)
else:
    dq_results_path = get_path(ENV_NAME, "metadata", config=CONFIG)
    lakehouse_table_write(dq_result, dq_results_path, "DQ_RESULTS", mode="append")
    if TARGET_KIND == "lakehouse":
        target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
        lakehouse_table_write(df_valid, target_path, TARGET_TABLE, mode=WRITE_MODE)
        lakehouse_table_write(df_quarantine, target_path, f"{TARGET_TABLE}_QUARANTINE", mode=WRITE_MODE)
    else:
        warehouse_write(df_valid, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)
        warehouse_write(df_quarantine, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=f"{TARGET_TABLE}_QUARANTINE", mode=WRITE_MODE, config=CONFIG)
assert_dq_passed(dq_result)


In [ ]:
if USE_SAMPLE_DATA:
    df_written = df_valid
else:
    if TARGET_KIND == "lakehouse":
        target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
        df_written = lakehouse_table_read(target_path, TARGET_TABLE)
    else:
        df_written = warehouse_read(env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, config=CONFIG)
if df_written.count() == 0:
    raise ValueError("Fail fast: target write produced zero rows.")


In [ ]:
output_profile = generate_metadata_profile(df_written, dataset_name=DATASET_NAME)
output_profile_rows = profile_dataframe_to_metadata(df_written, dataset_name=DATASET_NAME)


In [ ]:
dq_result_records = [r.asDict(recursive=True) for r in dq_result.collect()]
quality_records = build_quality_result_records(
    {"results": dq_result_records, "can_continue": True},
    run_id=RUN_ID,
    dataset_name=DATASET_NAME,
    table_name=SOURCE_TABLE,
    table_stage="source",
)
dataset_record = build_dataset_run_record(
    run_id=RUN_ID,
    dataset_name=DATASET_NAME,
    environment=ENV_NAME,
    source_table=SOURCE_TABLE,
    target_table=TARGET_TABLE,
    status="completed",
    started_at_utc=runtime_context.get("started_at_utc"),
    row_count_source=df_source.count(),
    row_count_output=df_standard.count(),
)


## Lineage placeholder (deterministic scanning)

Deterministic lineage scanning requires notebook code text as input.

- Fabric notebook runtime does not reliably provide `__file__`.
- Use a placeholder code string for now, or paste collected notebook code text.
- This will be replaced in a follow-up PR when a Fabric notebook-cell collector helper is added.


In [ ]:
NOTEBOOK_CODE_FOR_LINEAGE = None
if NOTEBOOK_CODE_FOR_LINEAGE:
    lineage = build_lineage_from_notebook_code(NOTEBOOK_CODE_FOR_LINEAGE, use_ai=False)
    lineage_steps = lineage.get("steps", [])
else:
    lineage_steps = [
        {"step": 1, "source": SOURCE_TABLE, "target": "df_source", "operation": "Read sample source table"},
        {"step": 2, "source": "df_source", "target": "df_transformed", "operation": "Normalize email, trim status, cast amount, derive event_date"},
        {"step": 3, "source": "df_transformed", "target": "df_valid and df_quarantine", "operation": "Apply approved contract DQ rules and split quarantine rows"},
        {"step": 4, "source": "df_valid", "target": TARGET_TABLE, "operation": "Write valid output"},
    ]
    lineage = {"steps": lineage_steps}
lineage_records = build_lineage_records(dataset_name=DATASET_NAME, run_id=RUN_ID, source_tables=[SOURCE_TABLE], target_table=TARGET_TABLE, transformation_steps=lineage_steps)
display(lineage_records)
print(build_lineage_handover_markdown(lineage))


In [ ]:
contract_records = build_contract_records(contract)
source_profile_records = profile_dataframe_to_metadata(df_source, dataset_name=DATASET_NAME)
output_profile_records = profile_dataframe_to_metadata(df_valid, dataset_name=DATASET_NAME)

dq_result_records = [r.asDict(recursive=True) for r in dq_result.collect()]
quality_records = build_quality_result_records({"results": dq_result_records, "can_continue": True}, run_id=RUN_ID, dataset_name=DATASET_NAME, table_name=SOURCE_TABLE, table_stage="source")
dataset_record = build_dataset_run_record(run_id=RUN_ID, dataset_name=DATASET_NAME, environment=ENV_NAME, source_table=SOURCE_TABLE, target_table=TARGET_TABLE, status="completed", started_at_utc=runtime_context.get("started_at_utc"), row_count_source=df_source.count(), row_count_output=df_valid.count())

handover_summary = {
    "RUN_ID": RUN_ID,
    "contract_id": contract.get("contract_id"),
    "contract status": contract.get("status"),
    "source row count": df_source.count(),
    "valid row count": df_valid.count(),
    "quarantine row count": df_quarantine.count(),
    "DQ rule count": len(rules),
    "DQ failed rule summary": dq_result_records,
    "target table": TARGET_TABLE,
    "quarantine table": f"{TARGET_TABLE}_QUARANTINE",
    "metadata outputs written": ["dataset_record", "quality_records", "source_profile_records", "output_profile_records", "contract_records", "lineage_records"],
    "lineage status": "parsed_notebook" if NOTEBOOK_CODE_FOR_LINEAGE else "manual_fallback",
}
print(handover_summary)
print("Lineage options: paste exported notebook source into NOTEBOOK_CODE_FOR_LINEAGE, or use manual lineage_steps fallback.")
